# Enhanced Turkish ODQA Demo

A compact, single-question walkthrough of the full pipeline:

1. retrieve top 100 passages for the original question;
2. rewrite the question with the LLM and retrieve top 100 again;
3. rerank the rewritten retrieval with the neural knowledge selector;
4. classify the question with adaptive retrieval and show the selected context budget.

This notebook follows the defaults in `scripts/run_full_pipeline_llm_selector_adaptive.sh`.

In [1]:
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

# Match scripts/run_full_pipeline_llm_selector_adaptive.sh defaults.
CUDA_DEVICE = "2"
GEMMA_PYTHON_BIN = "/cta/users/buse/miniconda3/envs/fsmodqa_gemma/bin/python"
FSMODQA_PYTHON_BIN = "/cta/users/buse/miniconda3/envs/fsmodqa_env/bin/python"
CHECKPOINT_DIR = ROOT / "checkpoint/final_fsmodqa_squad_tr_full_deavg"
TRAIN_DIR = ROOT / "odqa_data/fsmodqa_retrieval"
CORPUS_FILE = TRAIN_DIR / "corpus.jsonl"
QID_ALIASES = TRAIN_DIR / "squad_duplicate_qid_lookup.json"

REWRITE_MODEL = "google/gemma-2-2b-it"
RETRIEVER_MODEL = CHECKPOINT_DIR / "checkpoint-best"
READER_MODEL = CHECKPOINT_DIR / "checkpoint-best"
PASSAGE_EMBEDDINGS = str(CHECKPOINT_DIR / "encoding/passage_embedding_split*.pt")
SELECTOR_MODEL_DIR = CHECKPOINT_DIR / "neural_knowledge_selector"

# Read an existing trained adaptive model/policy. This notebook never trains adaptive retrieval.
PIPELINE_DIR = CHECKPOINT_DIR / "results/results_full_pipeline"
ADAPTIVE_DIR = PIPELINE_DIR / "adaptive_retrieval"
if not (ADAPTIVE_DIR / "adaptive_retrieval_classifier.pkl").exists():
    ADAPTIVE_DIR = ROOT / "checkpoint/adaptive_retrieval"

WORK_DIR = ROOT / "checkpoint/demo_notebook"
WORK_DIR.mkdir(parents=True, exist_ok=True)

TOP_N = 100
RETRIEVE_DEPTH = 100
RETRIEVAL_BATCH_SIZE = 128
ENCODE_BATCH_SIZE = 1
MAX_QUERY_LENGTH = 50
MAX_PASSAGE_LENGTH = 200
MAX_NEW_TOKENS = 128
SELECTOR_EVAL_BATCH_SIZE = 32
SELECTOR_MAX_LENGTH = 256
MATCH_MODE = "squad-prefix"

print(f"ROOT={ROOT}")
print(f"WORK_DIR={WORK_DIR}")
print(f"ADAPTIVE_DIR={ADAPTIVE_DIR}")

ROOT=/cta/users/buse/repos/turkish-openqa
WORK_DIR=/cta/users/buse/repos/turkish-openqa/checkpoint/demo_notebook
ADAPTIVE_DIR=/cta/users/buse/repos/turkish-openqa/checkpoint/final_fsmodqa_squad_tr_full_deavg/results/results_full_pipeline/adaptive_retrieval


In [7]:
import json
import os
import subprocess
import sys
from types import SimpleNamespace

import pandas as pd

sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "external/FSMODQA"))

import llm_expand_retrieve as llm_retrieval

os.environ["CUDA_VISIBLE_DEVICES"] = CUDA_DEVICE
os.environ.setdefault("MKL_THREADING_LAYER", "GNU")


DEMO_OUTPUT_ROOT = WORK_DIR.resolve()
READ_ONLY_INPUTS = {
    "retriever_model": RETRIEVER_MODEL,
    "reader_model": READER_MODEL,
    "selector_model_dir": SELECTOR_MODEL_DIR,
    "adaptive_dir": ADAPTIVE_DIR,
}


def is_relative_to(path, root):
    path = Path(path).resolve()
    root = Path(root).resolve()
    return path == root or root in path.parents


def assert_demo_output_path(path):
    path = Path(path)
    if not is_relative_to(path, DEMO_OUTPUT_ROOT):
        raise ValueError(f"Notebook outputs must stay under {DEMO_OUTPUT_ROOT}; refused: {path}")
    return path


def assert_read_only_inputs_exist():
    missing = []
    for name, input_path in READ_ONLY_INPUTS.items():
        input_path = Path(input_path)
        if not input_path.exists():
            missing.append(f"{name}: {input_path}")
    adaptive_model = ADAPTIVE_DIR / "adaptive_retrieval_classifier.pkl"
    adaptive_policy = ADAPTIVE_DIR / "policy.json"
    if not adaptive_model.exists():
        missing.append(f"adaptive_model: {adaptive_model}")
    if not adaptive_policy.exists():
        missing.append(f"adaptive_policy: {adaptive_policy}")
    if missing:
        raise FileNotFoundError("Missing trained read-only inputs:" + "".join(missing))
    print("Read-only trained inputs found. Notebook outputs are restricted to:", DEMO_OUTPUT_ROOT)


assert_read_only_inputs_exist()


def read_jsonl(path):
    rows = []
    with Path(path).open(encoding="utf-8") as handle:
        for line in handle:
            if line.strip():
                rows.append(json.loads(line))
    return rows


def write_jsonl(path, rows):
    path = assert_demo_output_path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as handle:
        for row in rows:
            handle.write(json.dumps(row, ensure_ascii=False) + "\n")


def load_corpus(path):
    corpus = {}
    for row in read_jsonl(path):
        pid = str(row.get("id") or row.get("docid"))
        corpus[pid] = row
    return corpus


corpus = load_corpus(CORPUS_FILE)
print(f"Loaded {len(corpus):,} passages")

Read-only trained inputs found. Notebook outputs are restricted to: /cta/users/buse/repos/turkish-openqa/checkpoint/demo_notebook
Loaded 1,095,020 passages


In [10]:
def retrieval_args(tag):
    return SimpleNamespace(
        retriever_model=str(RETRIEVER_MODEL),
        work_dir=WORK_DIR / tag,
        train_dir=TRAIN_DIR,
        corpus_file=CORPUS_FILE.name,
        encode_batch_size=ENCODE_BATCH_SIZE,
        max_query_length=MAX_QUERY_LENGTH,
        max_passage_length=MAX_PASSAGE_LENGTH,
        separate_joint_encoding=True,
        de_avg_pooling=True,
        add_lang_token=True,
        bf16=False,
        fp16=False,
        tf32=True,
    )


def run_encode_with_fsmodqa_env(args, query_file, encoded_file):
    assert_demo_output_path(args.work_dir)
    assert_demo_output_path(args.work_dir / "encode_output")
    encoded_file = assert_demo_output_path(encoded_file)
    encoded_file.parent.mkdir(parents=True, exist_ok=True)
    command = [
        FSMODQA_PYTHON_BIN,
        str(ROOT / "external/FSMODQA/encode.py"),
        "--model_name_or_path", args.retriever_model,
        "--output_dir", str((args.work_dir / "encode_output").resolve()),
        "--train_dir", str(args.train_dir.resolve()),
        "--corpus_file", args.corpus_file,
        "--query_file", str(query_file.resolve()),
        "--encode_is_qry",
        "--per_device_eval_batch_size", str(args.encode_batch_size),
        "--max_query_length", str(args.max_query_length),
        "--max_passage_length", str(args.max_passage_length),
        "--encoded_save_path", str(encoded_file.resolve()),
    ]
    if args.separate_joint_encoding:
        command.append("--separate_joint_encoding")
    if args.de_avg_pooling:
        command.append("--de_avg_pooling")
    if args.add_lang_token:
        command.append("--add_lang_token")
    if args.bf16:
        command.extend(["--bf16", "True"])
    if args.fp16:
        command.extend(["--fp16", "True"])
    if args.tf32:
        command.extend(["--tf32", "True"])

    print("Encoding query with", FSMODQA_PYTHON_BIN)
    subprocess.run(
        command,
        cwd=ROOT / "external/FSMODQA",
        env={**os.environ, "CUDA_VISIBLE_DEVICES": CUDA_DEVICE, "MKL_THREADING_LAYER": "GNU"},
        check=True,
    )


def retrieve_one(qid, question, tag):
    args = retrieval_args(tag)
    args.work_dir.mkdir(parents=True, exist_ok=True)
    query_file = args.work_dir / "query.jsonl"
    encoded_file = args.work_dir / "query_embedding.pt"
    ranking_file = args.work_dir / "top100.jsonl"
    rows = [{"id": qid, "question": question, "answers": ["placeholder"], "lang": "tr", "cl_answers": {}}]
    write_jsonl(query_file, rows)
    run_encode_with_fsmodqa_env(args, query_file, encoded_file)
    scores, pids, qids = llm_retrieval.search_inner_product(
        query_embeddings_file=encoded_file,
        passage_embedding_pattern=PASSAGE_EMBEDDINGS,
        depth=RETRIEVE_DEPTH,
        batch_size=RETRIEVAL_BATCH_SIZE,
        use_gpu=False,
    )
    ranking_rows = llm_retrieval.aggregate_rankings(scores, pids, qids, TOP_N)
    write_jsonl(ranking_file, ranking_rows)
    return ranking_file, ranking_rows[0]


def ranking_table(ranking_row, n=100):
    rows = []
    scores = ranking_row.get("scores") or []
    selector_scores = ranking_row.get("selector_scores") or []
    retriever_scores = ranking_row.get("retriever_scores") or []
    for rank, pid in enumerate(ranking_row.get("pids", [])[:n], start=1):
        passage = corpus.get(str(pid), {})
        rows.append(
            {
                "rank": rank,
                "pid": str(pid),
                "score": scores[rank - 1] if rank <= len(scores) else None,
                "selector_score": selector_scores[rank - 1] if rank <= len(selector_scores) else None,
                "retriever_score": retriever_scores[rank - 1] if rank <= len(retriever_scores) else None,
                "title": passage.get("title", ""),
                "text": passage.get("text", "")[:700],
            }
        )
    return pd.DataFrame(rows)


def show_top100(title, ranking_row):
    print(title)
    return ranking_table(ranking_row, TOP_N)


## 1. Ask A Question

Edit `QUESTION` and run the cells below.

In [44]:
QID = "demo-question-1"
QUESTION = "Nazım Hikmet nerede öldü?" #"Türkiye'nin ilk kadın pilotu kimdir?"
QUESTION

'Nazım Hikmet nerede öldü?'

## 2. Original Retrieval Top 100

In [45]:
original_ranking_file, original_ranking = retrieve_one(QID, QUESTION, "original")
original_passages = show_top100("Original question retrieval", original_ranking)

Encoding query with /cta/users/buse/miniconda3/envs/fsmodqa_env/bin/python


06/03/2026 08:51:15 - INFO - dataloader -   Loading Queries...
06/03/2026 08:51:15 - INFO - dataloader -   Loaded 1 Queries.
06/03/2026 08:51:15 - INFO - dataloader -   Query Example: ('Nazım Hikmet nerede öldü?', ['placeholder'], {}, 'tr')
06/03/2026 08:51:16 - INFO - __main__ -   Generate passage embeddings from 0 to 1
  0%|          | 0/1 [00:00<?, ?it/s]/cta/users/buse/miniconda3/envs/fsmodqa_env/lib/python3.10/site-packages/transformers/modeling_utils.py:884: FutureWarning: The `device` argument is deprecated and will be removed in v5 of Transformers.
  warnings.warn(
100%|██████████| 1/1 [00:00<00:00,  2.26it/s]
/cta/users/buse/repos/turkish-openqa/src/llm_expand_retrieve.py:459: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURIT

Original question retrieval


In [46]:
original_passages

,rank,pid,score,selector_score,retriever_score,title,text
0,1,wiki-102443-403627-0,939.600159,None,None,Nâzım Hikmet,"Nâzım Hikmet ( d. 14 Ocak 1902 ; Selanik, Osma..."
1,2,wiki-102443-403627-7,939.369629,None,None,Nâzım Hikmet,"Takrir - i Sükûn Kanunu aracılığıyla liberal, ..."
2,3,wiki-102443-403627-6,939.200989,None,None,Nâzım Hikmet,"inceledi ; Bagritski, Mayakovski, Selvinski, İ..."
3,4,wiki-102443-403627-14,938.993103,None,None,Nâzım Hikmet,çıkarılan genel af kanununun ertesi gün Resmi ...
4,5,wiki-2041-265505-1,938.882385,None,None,29 Ekim,"1924 - Milletler Cemiyeti Konseyinde, Türkiye ..."
...,...,...,...,...,...,...,...
95,96,squad-56f8f4539e9bad19000a0773-4392-0,938.522583,None,None,Yakın Doğu,Yakın Doğu'nun ölümünden itibaren yeni uluslar...
96,97,wiki-487090-23250-1,938.520508,None,None,Inayat Khan,", Vilayat ( 1916 ), Hidayat ( 1917 ) ve Khair ..."
97,98,wiki-18654-134089-0,938.520447,None,None,1138,"Olaylar Doğumlar Selahaddin Eyyubi, Mısır ve S..."
98,99,wiki-1308873-438615-8,938.520203,None,None,Nimetullah Hafız,"Âşığı Âşık Hivzi, Sesler, 200, Kasim 1985, Üsk..."


## 3. Rewrite The Question And Retrieve Top 100

In [47]:
rewrite_tokenizer, rewrite_model = llm_retrieval.load_rewrite_model(REWRITE_MODEL)
rewrite_rows, rewrite_debug = llm_retrieval.build_rewritten_queries(
    input_rows=[{"id": QID, "question": QUESTION, "answers": ["placeholder"], "lang": "tr"}],
    tokenizer=rewrite_tokenizer,
    model=rewrite_model,
    max_new_tokens=MAX_NEW_TOKENS,
    temperature=0.0,
)
REWRITTEN_QUESTION = rewrite_rows[0]["question"]
print("Original: ", QUESTION)
print("Rewritten:", REWRITTEN_QUESTION)
display(pd.DataFrame(rewrite_debug))

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

Generating query rewrites: 100%|██████████| 1/1 [00:00<00:00,  4.90it/s]

Original:  Nazım Hikmet nerede öldü?
Rewritten: Nazım Hikmet'in ölüm yeri


,id,question,rewrite_generation,rewritten_query
0,demo-question-1,Nazım Hikmet nerede öldü?,Nazım Hikmet'in ölüm yeri,Nazım Hikmet'in ölüm yeri


In [48]:
rewritten_ranking_file, rewritten_ranking = retrieve_one(QID, REWRITTEN_QUESTION, "rewritten")
show_top100("Rewritten question retrieval", rewritten_ranking)

Encoding query with /cta/users/buse/miniconda3/envs/fsmodqa_env/bin/python


06/03/2026 08:51:53 - INFO - dataloader -   Loading Queries...
06/03/2026 08:51:53 - INFO - dataloader -   Loaded 1 Queries.
06/03/2026 08:51:53 - INFO - dataloader -   Query Example: ("Nazım Hikmet'in ölüm yeri", ['placeholder'], {}, 'tr')
06/03/2026 08:51:55 - INFO - __main__ -   Generate passage embeddings from 0 to 1
  0%|          | 0/1 [00:00<?, ?it/s]/cta/users/buse/miniconda3/envs/fsmodqa_env/lib/python3.10/site-packages/transformers/modeling_utils.py:884: FutureWarning: The `device` argument is deprecated and will be removed in v5 of Transformers.
  warnings.warn(
100%|██████████| 1/1 [00:00<00:00,  2.21it/s]
/cta/users/buse/repos/turkish-openqa/src/llm_expand_retrieve.py:459: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURIT

Rewritten question retrieval


,rank,pid,score,selector_score,retriever_score,title,text
0,1,wiki-102443-403627-0,936.291260,None,None,Nâzım Hikmet,"Nâzım Hikmet ( d. 14 Ocak 1902 ; Selanik, Osma..."
1,2,wiki-102443-403627-7,935.772583,None,None,Nâzım Hikmet,"Takrir - i Sükûn Kanunu aracılığıyla liberal, ..."
2,3,wiki-1127374-300273-1,935.756104,None,None,Karamanlı Nizamî,"olduğundan, onu başkenti İstanbul'a davet etmi..."
3,4,wiki-425330-419765-1,935.645142,None,None,Arif Hikmet Paşa,Vakası'ndan sonra Meclis - i Millî'nin II. Abd...
4,5,wiki-93555-403311-0,935.598022,None,None,Ömer Hayyam,Gıyaseddin Ebu'l - Feth Ömer ibni İbrahim Nişa...
...,...,...,...,...,...,...,...
95,96,wiki-2646598-353118-3,935.147217,None,None,Wilhelm Kienzl,"ölçüde ihmal edilmiş olsa da, şarkıların geniş..."
96,97,wiki-340542-151454-38,935.146484,None,None,Yaratılış Kitabı,ile barışçıl bir şekilde karşılaşır ( 33 : 1 )...
97,98,wiki-10532-266248-38,935.145020,None,None,Timur,Halep'te yaklaşık 15 gün kadar kaldı. Şehir ya...
98,99,wiki-54465-3663-3,935.142822,None,None,Nişancı Mehmed Paşa,"görevine nakledildi. Mısır valiliğinde, kısa b..."


## 4. Knowledge Selector Reranking

In [49]:
import subprocess

selector_query_file = WORK_DIR / "rewritten_selector.query.jsonl"
selector_output_file = assert_demo_output_path(WORK_DIR / "rewritten_selector_reranked.jsonl")
write_jsonl(selector_query_file, [{"id": QID, "question": REWRITTEN_QUESTION, "answers": ["placeholder"], "lang": "tr"}])
assert Path(SELECTOR_MODEL_DIR).exists(), f"Missing trained selector model: {SELECTOR_MODEL_DIR}"

selector_cmd = [
    FSMODQA_PYTHON_BIN,
    str(ROOT / "src/apply_fsmodqa_neural_selector.py"),
    "--model-dir", str(SELECTOR_MODEL_DIR),
    "--ranking", str(rewritten_ranking_file),
    "--queries", str(selector_query_file),
    "--corpus", str(CORPUS_FILE),
    "--output", str(selector_output_file),
    "--n-context", str(TOP_N),
    "--eval-batch-size", str(SELECTOR_EVAL_BATCH_SIZE),
    "--max-length", str(SELECTOR_MAX_LENGTH),
]

print("Running selector with", FSMODQA_PYTHON_BIN)
subprocess.run(
    selector_cmd,
    cwd=ROOT,
    env={**os.environ, "CUDA_VISIBLE_DEVICES": CUDA_DEVICE, "MKL_THREADING_LAYER": "GNU"},
    check=True,
)

selector_ranking = read_jsonl(selector_output_file)[0]
show_top100("Knowledge selector reranked retrieval", selector_ranking)


Running selector with /cta/users/buse/miniconda3/envs/fsmodqa_env/bin/python
[knowledge-selector] Importing torch and transformers
[knowledge-selector] Loading corpus: /cta/users/buse/repos/turkish-openqa/odqa_data/fsmodqa_retrieval/corpus.jsonl
[knowledge-selector] Loading corpus passages from /cta/users/buse/repos/turkish-openqa/odqa_data/fsmodqa_retrieval/corpus.jsonl
[knowledge-selector] Loaded 1095020 corpus passages from /cta/users/buse/repos/turkish-openqa/odqa_data/fsmodqa_retrieval/corpus.jsonl
[knowledge-selector] Loading queries from /cta/users/buse/repos/turkish-openqa/checkpoint/demo_notebook/rewritten_selector.query.jsonl
[knowledge-selector] Loaded 1 queries from /cta/users/buse/repos/turkish-openqa/checkpoint/demo_notebook/rewritten_selector.query.jsonl
[knowledge-selector] Building selector pairs from /cta/users/buse/repos/turkish-openqa/checkpoint/demo_notebook/rewritten/top100.jsonl
[knowledge-selector] Loaded 1 ranking records from /cta/users/buse/repos/turkish-open

,rank,pid,score,selector_score,retriever_score,title,text
0,1,wiki-102443-403627-0,0.701502,0.573574,936.291260,Nâzım Hikmet,"Nâzım Hikmet ( d. 14 Ocak 1902 ; Selanik, Osma..."
1,2,wiki-68138-402346-5,0.481264,0.588899,935.406860,Bedia Muvahhit,Nâzım Hikmet'in yazıp Muhsin Ertuğrul'un yönet...
2,3,wiki-2472385-215479-21,0.383285,0.537917,935.168335,Nâzım Hikmet şiirleri listesi,"Akşam Oldu ( 1919, Nişantaşı ) Acılarımdan ( 1..."
3,4,wiki-2472386-348187-0,0.351195,0.464955,935.241028,Nâzım Hikmet eserleri listesi,"Türk şair, oyun yazarı, romancı ve anı yazarı ..."
4,5,wiki-102443-403627-1,0.318227,0.390237,935.315063,Nâzım Hikmet,soyadı yerine babasının adını kullanarak hep N...
...,...,...,...,...,...,...,...
95,96,wiki-2648039-353210-1,0.018491,0.016670,935.168640,Dengizik,savaşta öldürüldü ve kesik başı Bizans başkent...
96,97,wiki-2905323-361529-2,0.018215,0.022492,935.151978,Nakam,"Abba Kovner, Ponary katliamı ve Majdanek'teki ..."
97,98,wiki-54465-3663-3,0.015156,0.021537,935.142822,Nişancı Mehmed Paşa,"görevine nakledildi. Mısır valiliğinde, kısa b..."
98,99,wiki-29820-400429-3,0.014266,0.020380,935.142517,Nefertiti,Akhenaten'in ölümü ile Tutankhamun'un tahta çı...


## 5. Adaptive Retrieval Classification

In [50]:
adaptive_model_path = ADAPTIVE_DIR / "adaptive_retrieval_classifier.pkl"
adaptive_policy_path = ADAPTIVE_DIR / "policy.json"
assert adaptive_model_path.exists(), f"Missing adaptive model: {adaptive_model_path}"
assert adaptive_policy_path.exists(), f"Missing adaptive policy: {adaptive_policy_path}"

adaptive_query_file = WORK_DIR / "adaptive.query.jsonl"
adaptive_apply_dir = assert_demo_output_path(WORK_DIR / "adaptive_apply")
write_jsonl(adaptive_query_file, [{"id": QID, "question": REWRITTEN_QUESTION, "answers": ["placeholder"], "lang": "tr"}])

adaptive_cmd = [
    GEMMA_PYTHON_BIN,
    str(ROOT / "src/train_adaptive_retrieval.py"),
    "--mode", "apply",
    "--model", str(adaptive_model_path),
    "--policy", str(adaptive_policy_path),
    "--test-rankings", str(selector_output_file),
    "--test-queries", str(adaptive_query_file),
    "--corpus", str(CORPUS_FILE),
    "--qid-aliases", str(QID_ALIASES),
    "--output-dir", str(adaptive_apply_dir),
    "--match-mode", MATCH_MODE,
]

assert "--mode" in adaptive_cmd and adaptive_cmd[adaptive_cmd.index("--mode") + 1] == "apply"
assert "--train-rankings" not in adaptive_cmd and "--validation-rankings" not in adaptive_cmd
print("Running adaptive classification in apply-only mode with", GEMMA_PYTHON_BIN)
subprocess.run(
    adaptive_cmd,
    cwd=ROOT,
    env={**os.environ, "CUDA_VISIBLE_DEVICES": CUDA_DEVICE, "MKL_THREADING_LAYER": "GNU"},
    check=True,
)

with (adaptive_apply_dir / "test_adaptive.json").open(encoding="utf-8") as handle:
    selected_rows = json.load(handle)
selected = selected_rows[0]

with adaptive_policy_path.open(encoding="utf-8") as handle:
    adaptive_policy = json.load(handle)

print("Adaptive label:", selected["adaptive_label"])
print("Adaptive k:", selected["adaptive_k"])
print("Policy:", adaptive_policy)

adaptive_ranking = {"qid": selected["id"], "pids": selected["pids"], "scores": selected.get("scores", [])}
classification_row = {
    "id": selected["id"],
    "question": selected["question"],
    "predicted_label": selected.get("adaptive_label", "hard"),
    "adaptive_k": selected.get("adaptive_k"),
    "first_hit_rank_before_budget": selected.get("original_first_hit_rank"),
    "first_hit_rank_after_budget": selected.get("first_hit_rank"),
}
display(pd.DataFrame([classification_row]))
show_top100("Adaptive selected passages", adaptive_ranking)


Running adaptive classification in apply-only mode with /cta/users/buse/miniconda3/envs/fsmodqa_gemma/bin/python
[adaptive-retrieval] Loading corpus: /cta/users/buse/repos/turkish-openqa/odqa_data/fsmodqa_retrieval/corpus.jsonl
[adaptive-retrieval] Loading adaptive classifier: /cta/users/buse/repos/turkish-openqa/checkpoint/final_fsmodqa_squad_tr_full_deavg/results/results_full_pipeline/adaptive_retrieval/adaptive_retrieval_classifier.pkl
[adaptive-retrieval] Loading adaptive policy: /cta/users/buse/repos/turkish-openqa/checkpoint/final_fsmodqa_squad_tr_full_deavg/results/results_full_pipeline/adaptive_retrieval/policy.json
[adaptive-retrieval] Building test examples for apply-only adaptive retrieval
[adaptive-retrieval] Saved apply-only adaptive outputs to /cta/users/buse/repos/turkish-openqa/checkpoint/demo_notebook/adaptive_apply
[adaptive-retrieval] Adaptive test avg contexts: 5.00
Adaptive label: easy
Adaptive k: 5
Policy: {'easy_k': 5, 'medium_k': 25, 'hard_k': 100, 'recall': 0.7

,id,question,predicted_label,adaptive_k,first_hit_rank_before_budget,first_hit_rank_after_budget
0,demo-question-1,Nazım Hikmet'in ölüm yeri,easy,5,None,None


Adaptive selected passages


,rank,pid,score,selector_score,retriever_score,title,text
0,1,wiki-102443-403627-0,0.701502,None,None,Nâzım Hikmet,"Nâzım Hikmet ( d. 14 Ocak 1902 ; Selanik, Osma..."
1,2,wiki-68138-402346-5,0.481264,None,None,Bedia Muvahhit,Nâzım Hikmet'in yazıp Muhsin Ertuğrul'un yönet...
2,3,wiki-2472385-215479-21,0.383285,None,None,Nâzım Hikmet şiirleri listesi,"Akşam Oldu ( 1919, Nişantaşı ) Acılarımdan ( 1..."
3,4,wiki-2472386-348187-0,0.351195,None,None,Nâzım Hikmet eserleri listesi,"Türk şair, oyun yazarı, romancı ve anı yazarı ..."
4,5,wiki-102443-403627-1,0.318227,None,None,Nâzım Hikmet,soyadı yerine babasının adını kullanarak hep N...


## 6. Final Reader Answer

In [51]:
import subprocess

reader_ranking_file = assert_demo_output_path(WORK_DIR / "adaptive_selected_rankings.jsonl")
reader_predictions_file = assert_demo_output_path(WORK_DIR / "reader_predictions.json")
write_jsonl(reader_ranking_file, [adaptive_ranking])

n_passages = max(1, len(adaptive_ranking["pids"]))
reader_cmd = [
    FSMODQA_PYTHON_BIN,
    "test_reader.py",
    "--output_dir", str(CHECKPOINT_DIR),
    "--model_name_or_path", str(READER_MODEL),
    "--output_path", str(reader_predictions_file),
    "--train_dir", str(TRAIN_DIR),
    "--train_path", str(reader_ranking_file),
    "--corpus_file", CORPUS_FILE.name,
    "--query_file", str(adaptive_query_file),
    "--per_device_eval_batch_size", "1",
    "--train_n_passages", str(n_passages),
    "--max_query_length", "50",
    "--max_passage_length", "200",
    "--max_query_passage_length", "250",
    "--max_answer_length", "50",
    "--separate_joint_encoding",
    "--de_avg_pooling",
    "--add_lang_token",
    "--bf16", "False",
    "--tf32", "True",
]

print("Running reader with", n_passages, "passages using", FSMODQA_PYTHON_BIN)
subprocess.run(
    reader_cmd,
    cwd=ROOT / "external/FSMODQA",
    env={**os.environ, "CUDA_VISIBLE_DEVICES": CUDA_DEVICE, "MKL_THREADING_LAYER": "GNU"},
    check=True,
)

with reader_predictions_file.open(encoding="utf-8") as handle:
    reader_predictions = json.load(handle)

final_answer = reader_predictions.get(QID, "")
print("Question:", QUESTION)
print("Rewritten question:", REWRITTEN_QUESTION)
print("Adaptive label:", selected["adaptive_label"])
print("Adaptive k:", selected["adaptive_k"])
print("Final answer:", final_answer)
display(pd.DataFrame([{"qid": QID, "question": QUESTION, "rewritten_question": REWRITTEN_QUESTION, "adaptive_label": selected["adaptive_label"], "adaptive_k": selected["adaptive_k"], "prediction": final_answer}]))


Running reader with 5 passages using /cta/users/buse/miniconda3/envs/fsmodqa_env/bin/python


06/03/2026 08:52:28 - WARNING - __main__ -   Process rank: 0, device: cuda:0, n_gpu: 1, distributed training: True, 16-bits training: False
06/03/2026 08:52:28 - INFO - __main__ -   Training/evaluation parameters BiEncoderTrainingArguments(
_n_gpu=1,
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_pin_memory=True,
ddp_backend=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
de_avg_pooling=True,
debug=False,
deepspeed=None,
disable_tqdm=False,
distillation_start_steps=3000,
distributed_port=None,
do_encode=False,
do_eval=False,
do_predict=False,
do_train=False,
e2e_training=False,
eval_accumulation_steps=None,
eval_at_start=False,
eval_delay=0,
eval_on_mkqa=False,
eval_on_test=False,
eval_steps=None,
evaluation_strategy=no,
fp16=False,
fp16_backend=auto,
fp16_full_eval=False,
fp16_opt_level=O

Question: Nazım Hikmet nerede öldü?
Rewritten question: Nazım Hikmet'in ölüm yeri
Adaptive label: easy
Adaptive k: 5
Final answer: Moskova


,qid,question,rewritten_question,adaptive_label,adaptive_k,prediction
0,demo-question-1,Nazım Hikmet nerede öldü?,Nazım Hikmet'in ölüm yeri,easy,5,Moskova
